# MagnetDB — Raw Data Processing for Knowledge Graph
**Project:** MetaboGlue / EvoAge KG  |  **Contributor:** Arushi

All input files are read from `BASE_PATH`.  
All output files are written to `OUT_PATH` (main) or `OUT_PATH + "EvOlf/"` (EvOlf).


## 0. Path Configuration

In [1]:
import os

# ── Change only these two lines to relocate all I/O ──────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"   # all raw input files live here
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/"    # all processed CSVs will be written here
# ─────────────────────────────────────────────────────────────────────────────

# Sub-paths derived automatically – do not edit
EVOLF_IN_PATH  = os.path.join(BASE_PATH, "evolf")
EVOLF_OUT_PATH = os.path.join(OUT_PATH,  "evolf")

# Ensure output directories exist
os.makedirs(OUT_PATH,       exist_ok=True)
os.makedirs(EVOLF_OUT_PATH, exist_ok=True)

print("BASE_PATH :", BASE_PATH)
print("OUT_PATH  :", OUT_PATH)


BASE_PATH : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
OUT_PATH  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/


## 1. Imports

In [2]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from glob import glob


## 2. Shared Helper Function


In [3]:
def map_head_id(head_value):
    """Map a compound name (possibly '::'-separated) to a PubChem CID."""
    for name in str(head_value).split("::"):
        if name in Pubchem_Syn_fil_dict:
            return Pubchem_Syn_fil_dict[name]
    return np.nan


## 3. Reference Dictionaries (Mapping Files)

### 3.1 PubChem Synonym → CID

In [4]:
Pubchem_Syn_fil = pd.read_csv(
    os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered"),
    sep="\t", header=None
)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
del Pubchem_Syn_fil
print(f"Pubchem_Syn_fil_dict loaded: {len(Pubchem_Syn_fil_dict):,} entries")


Pubchem_Syn_fil_dict loaded: 102,877,691 entries


### 3.2 PubChem CID → SMILES / IUPAC

In [5]:
Pubchem = pd.read_pickle(os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl"))
Pubchem_CID_Smile_Dict  = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_SMILES"]))
Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_IUPAC_NAME"]))
del Pubchem
print(f"Pubchem_CID_Smile_Dict : {len(Pubchem_CID_Smile_Dict):,} entries")
print(f"Pubchem_IUPAC_CID_Dict : {len(Pubchem_IUPAC_CID_Dict):,} entries")


Pubchem_CID_Smile_Dict : 119,177,440 entries
Pubchem_IUPAC_CID_Dict : 119,177,440 entries


### 3.3 DrugBank ↔ PubChem CID

In [6]:
pubchem_db = pd.read_csv(os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv"))
pubchem_db = pubchem_db[~pubchem_db["DRUGBANK_ID"].isna()]
DB2PC_dict  = dict(zip(pubchem_db["DRUGBANK_ID"], pubchem_db["ID"]))   # DrugBank → PubChem
DB2Pub_dict = dict(zip(pubchem_db["ID"],          pubchem_db["DRUGBANK_ID"]))  # PubChem → DrugBank
del pubchem_db
print(f"DB2PC_dict : {len(DB2PC_dict):,} entries")


/tmp/ipykernel_2250426/285543507.py:1: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem_db = pd.read_csv(os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv"))


DB2PC_dict : 10,702 entries


### 3.4 UniProt RecName ↔ AC

In [7]:
Uniprot_Recname = pd.read_csv(os.path.join(BASE_PATH, "databases_for_mapping/uniprot/Uniprot_ID_Recname.csv"))
Uniprot_Recname_dict    = dict(zip(Uniprot_Recname["RecName"], Uniprot_Recname["AC"]))
Uniprot_Recname_dictRev = dict(zip(Uniprot_Recname["AC"],      Uniprot_Recname["RecName"]))
del Uniprot_Recname
print(f"Uniprot_Recname_dict : {len(Uniprot_Recname_dict):,} entries")


Uniprot_Recname_dict : 149,154 entries


In [15]:
Uniprot_Recname_dictRev

{'Q6GZX4': 'Putative transcription factor 001R',
 'Q6GZX3': 'Uncharacterized protein 002L',
 'Q197F8': 'Uncharacterized protein 002R',
 'Q197F7': 'Uncharacterized protein 003L',
 'Q6GZX2': 'Uncharacterized protein 3R',
 'Q6GZX1': 'Uncharacterized protein 004R',
 'Q197F5': 'Uncharacterized protein 005L',
 'Q6GZX0': 'Uncharacterized protein 005R',
 'Q91G88': 'Putative KilA-N domain-containing protein 006L',
 'Q6GZW9': 'Uncharacterized protein 006R',
 'Q6GZW8': 'Uncharacterized protein 007R',
 'Q197F3': 'Uncharacterized protein 007R',
 'Q197F2': 'Uncharacterized protein 008L',
 'Q6GZW6': 'Putative helicase 009L',
 'Q91G85': 'Uncharacterized protein 009R',
 'Q6GZW5': 'Uncharacterized protein 010R',
 'Q197E9': 'Uncharacterized protein 011L',
 'Q6GZW4': 'Uncharacterized protein 011R',
 'Q6GZW3': 'Uncharacterized protein 012L',
 'Q197E7': 'Uncharacterized protein IIV3-013L',
 'Q6GZW2': 'Uncharacterized protein 013R',
 'Q6GZW1': 'Uncharacterized protein 014R',
 'Q6GZW0': 'Uncharacterized prote

In [14]:
Uniprot_Recname_dict

{'Putative transcription factor 001R': 'Q6GZX4',
 'Uncharacterized protein 002L': 'Q6GZX3',
 'Uncharacterized protein 002R': 'Q197F8',
 'Uncharacterized protein 003L': 'Q197F7',
 'Uncharacterized protein 3R': 'Q6GZX2',
 'Uncharacterized protein 004R': 'Q197F6',
 'Uncharacterized protein 005L': 'Q197F5',
 'Uncharacterized protein 005R': 'Q6GZX0',
 'Putative KilA-N domain-containing protein 006L': 'Q91G88',
 'Uncharacterized protein 006R': 'Q6GZW9',
 'Uncharacterized protein 007R': 'Q197F3',
 'Uncharacterized protein 008L': 'Q197F2',
 'Putative helicase 009L': 'Q6GZW6',
 'Uncharacterized protein 009R': 'Q91G85',
 'Uncharacterized protein 010R': 'Q6GZW5',
 'Uncharacterized protein 011L': 'Q197E9',
 'Uncharacterized protein 011R': 'Q6GZW4',
 'Uncharacterized protein 012L': 'Q6GZW3',
 'Uncharacterized protein IIV3-013L': 'Q197E7',
 'Uncharacterized protein 013R': 'Q6GZW2',
 'Uncharacterized protein 014R': 'Q6GZW1',
 'Uncharacterized protein 015R': 'Q197E5',
 'Uncharacterized protein 017L': 

### 3.5 NCBI Gene dictionaries

In [8]:
ENS_2NCBI = pd.read_csv(os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv"))
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI["name"].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI["name"], ENS_2NCBI["initial_alias"]))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info"), sep="\t")
NCBI_ALL_GENE["ENSEMBLE_ID"] = NCBI_ALL_GENE["Symbol"].map(NCBI_2ENS__dict)

NCBI_ALL_GENEname_dict     = dict(zip(NCBI_ALL_GENE["GeneID"],      NCBI_ALL_GENE["description"]))
NCBI_ALL_GENEIDname_dict   = dict(zip(NCBI_ALL_GENE["GeneID"],      NCBI_ALL_GENE["Symbol"]))
NCBI_Synbol_GENEname_dict  = dict(zip(NCBI_ALL_GENE["Symbol"],      NCBI_ALL_GENE["description"]))
NCBI_GENEname__Symbol_dict = dict(zip(NCBI_ALL_GENE["description"], NCBI_ALL_GENE["Symbol"]))
del NCBI_ALL_GENE
print(f"NCBI_Synbol_GENEname_dict  : {len(NCBI_Synbol_GENEname_dict):,} entries")
print(f"NCBI_GENEname__Symbol_dict : {len(NCBI_GENEname__Symbol_dict):,} entries")


NCBI_Synbol_GENEname_dict  : 193,352 entries
NCBI_GENEname__Symbol_dict : 190,643 entries


## 4. BindingDB

In [17]:
species_list = [
    "Drosophila melanogaster", "Mus musculus", "Saccharomyces cerevisiae",
    "Caenorhabditis elegans", "Danio rerio", "Homo sapiens", "Humans",
]

Binding = pd.read_csv(os.path.join(BASE_PATH, "bindingdb/BindingDB_All_Column_3D.csv"))
print(f"Raw BindingDB rows: {len(Binding):,}")

Binding = Binding[
    Binding["Target Source Organism According to Curator or DataSource"].isin(species_list)
]
Binding = Binding[~Binding["UniProt (SwissProt) Primary ID of Target Chain"].isna()]
print(f"After species + UniProt filter: {len(Binding):,}")

Binding.rename(columns={
    "BindingDB Ligand Name":                               "Head",
    "UniProt (SwissProt) Primary ID of Target Chain":     "Tail",
    "Target Source Organism According to Curator or DataSource": "Species",
    "UniProt (SwissProt) Recommended Name of Target Chain": "Tail_Name",
    "UniProt (SwissProt) Entry Name of Target Chain":       "Tail_Symbol",
    "Ligand InChI":                                         "Head_Inchi",
}, inplace=True)

Binding["Relation"]        = "ChemicalEntity_Protein"
Binding["Head_type"]       = "ChemicalEntity"
Binding["Tail_type"]       = "Protein"
Binding["Relation_Source"] = "BindingDB_MetaboGlue"

Binding = Binding[[
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "Relation_Source", "Species", "Head_Inchi", "Tail_Name", "Tail_Symbol",
]]
Binding


/tmp/ipykernel_2250426/308444149.py:6: DtypeWarning: Columns (11,13,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  Binding = pd.read_csv(os.path.join(BASE_PATH, "bindingdb/BindingDB_All_Column_3D.csv"))


Raw BindingDB rows: 422,172
After species + UniProt filter: 226,366


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_Inchi,Tail_Name,Tail_Symbol
0,"6-[(4R,5S,6S,7R)-4,7-dibenzyl-3-(5-carboxypent...",ChemicalEntity_Protein,P35968,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C22H24BrFN4O2/c1-28-7-5-14(6-8-28)12-...,Vascular endothelial growth factor receptor 2,VGFR2_HUMAN
96,"pyrimidine-4-carboxamide, 119",ChemicalEntity_Protein,P0DMS8,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C16H18N6O2/c1-3-22-7-6-18-14(22)9-19-...,Adenosine receptor A3,AA3R_HUMAN
139,"US10112907, Example 00033::US10206907, Compoun...",ChemicalEntity_Protein,P52333,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C21H23N5O3S/c27-20(17-8-9-17)23-21-22...,Tyrosine-protein kinase JAK3,JAK3_HUMAN
141,"US9670213, Compound 005 8-(3-aminophenyl)-2-(4...",ChemicalEntity_Protein,P36888,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C19H16N6O2/c1-27-15-7-5-13(6-8-15)23-...,Receptor-type tyrosine-protein kinase FLT3,FLT3_HUMAN
194,5-({[(5-{[(2S)-1-carboxy-4-{[(2-chlorophenyl)m...,ChemicalEntity_Protein,P42575,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C27H27ClN2O7S2/c1-30(12-16-6-8-22(31)...,Caspase-2,CASP2_HUMAN
...,...,...,...,...,...,...,...,...,...,...
422157,CHEMBL4207014,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,"InChI=1S/C19H15N7O2S/c1-29(27,28)14-4-2-13(3-5...",Serine/threonine-protein kinase ATR,ATR_HUMAN
422158,CHEMBL4207205,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C19H17N9S/c20-16-14-15(10-5-9-3-4-22-...,Serine/threonine-protein kinase ATR,ATR_HUMAN
422159,CHEMBL4210861,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C21H17N7/c22-19-17-18(15-7-14-5-6-23-...,Serine/threonine-protein kinase ATR,ATR_HUMAN
422161,3-[2-(4-tert-butylphenoxy)acetamido]benzoic ac...,ChemicalEntity_Protein,Q99814,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,"InChI=1S/C19H21NO4/c1-19(2,3)14-7-9-16(10-8-14...",Endothelial PAS domain-containing protein 1,EPAS1_HUMAN


In [18]:
# Bug 1 fixed: reuse shared map_head_id() defined in Section 2
tqdm.pandas(desc="BindingDB – mapping Head → PubChem CID")
Binding["Head_ID"] = Binding["Head"].progress_apply(map_head_id)
Binding["Head_ID"] = Binding["Head_ID"].astype(str).str.replace(r"\.0$", "", regex=True)
Binding = Binding[Binding["Head_ID"].notna() & (Binding["Head_ID"] != "nan")]
print(f"After CID mapping: {len(Binding):,}")
Binding

BindingDB – mapping Head → PubChem CID: 100%|██████████| 226366/226366 [00:00<00:00, 350891.11it/s]


After CID mapping: 221,249


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_Inchi,Tail_Name,Tail_Symbol,Head_ID
0,"6-[(4R,5S,6S,7R)-4,7-dibenzyl-3-(5-carboxypent...",ChemicalEntity_Protein,P35968,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C22H24BrFN4O2/c1-28-7-5-14(6-8-28)12-...,Vascular endothelial growth factor receptor 2,VGFR2_HUMAN,3009304
96,"pyrimidine-4-carboxamide, 119",ChemicalEntity_Protein,P0DMS8,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C16H18N6O2/c1-3-22-7-6-18-14(22)9-19-...,Adenosine receptor A3,AA3R_HUMAN,44520983
139,"US10112907, Example 00033::US10206907, Compoun...",ChemicalEntity_Protein,P52333,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C21H23N5O3S/c27-20(17-8-9-17)23-21-22...,Tyrosine-protein kinase JAK3,JAK3_HUMAN,49831257
141,"US9670213, Compound 005 8-(3-aminophenyl)-2-(4...",ChemicalEntity_Protein,P36888,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C19H16N6O2/c1-27-15-7-5-13(6-8-15)23-...,Receptor-type tyrosine-protein kinase FLT3,FLT3_HUMAN,72712978
194,5-({[(5-{[(2S)-1-carboxy-4-{[(2-chlorophenyl)m...,ChemicalEntity_Protein,P42575,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C27H27ClN2O7S2/c1-30(12-16-6-8-22(31)...,Caspase-2,CASP2_HUMAN,5327301
...,...,...,...,...,...,...,...,...,...,...,...
422157,CHEMBL4207014,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,"InChI=1S/C19H15N7O2S/c1-29(27,28)14-4-2-13(3-5...",Serine/threonine-protein kinase ATR,ATR_HUMAN,145977670
422158,CHEMBL4207205,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C19H17N9S/c20-16-14-15(10-5-9-3-4-22-...,Serine/threonine-protein kinase ATR,ATR_HUMAN,145977679
422159,CHEMBL4210861,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,InChI=1S/C21H17N7/c22-19-17-18(15-7-14-5-6-23-...,Serine/threonine-protein kinase ATR,ATR_HUMAN,118025908
422161,3-[2-(4-tert-butylphenoxy)acetamido]benzoic ac...,ChemicalEntity_Protein,Q99814,ChemicalEntity,Protein,BindingDB_MetaboGlue,Homo sapiens,"InChI=1S/C19H21NO4/c1-19(2,3)14-7-9-16(10-8-14...",Endothelial PAS domain-containing protein 1,EPAS1_HUMAN,1363752


In [19]:
set(Binding['Species'])

Binding[Binding['Species'] == 'Danio rerio']

,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_Inchi,Tail_Name,Tail_Symbol,Head_ID
399850,CHEMBL2331570,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,"InChI=1S/C19H13F3N6O2/c20-19(21,22)11-2-1-3-13...",Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575543
399851,CHEMBL2332849,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,"InChI=1S/C19H13F3N6O2/c20-19(21,22)11-1-3-12(4...",Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575213
399852,CHEMBL2332871,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,"InChI=1S/C19H12ClF3N6O2/c20-14-7-10(19(21,22)2...",Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575641
399853,CHEMBL2332870,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,"InChI=1S/C20H12F6N6O2/c21-19(22,23)10-5-11(20(...",Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575640
399854,CHEMBL2332869,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,"InChI=1S/C19H13F3N6O3/c20-19(21,22)31-14-3-1-2...",Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575546
399855,CHEMBL2332868,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,InChI=1S/C18H13FN6O2/c19-11-2-1-3-13(8-11)24-1...,Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575545
399856,CHEMBL2332867,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,InChI=1S/C21H20N6O2/c1-13(2)14-4-3-5-16(10-14)...,Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575544
399857,CHEMBL2332866,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,InChI=1S/C22H21N7O3/c30-22(26-15-1-5-17(6-2-15...,Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575542
399858,CHEMBL2332865,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,InChI=1S/C18H13FN6O2/c19-11-1-3-12(4-2-11)23-1...,Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71575541
399859,CHEMBL2332864,ChemicalEntity_Protein,Q5GIT4,ChemicalEntity,Protein,BindingDB_MetaboGlue,Danio rerio,InChI=1S/C19H22N6O2/c26-19(20-10-13-4-2-1-3-5-...,Vascular endothelial growth factor receptor 2,VGFR2_DANRE,71716376


In [20]:
# Swap Head ↔ Head_ID so canonical CID is in the Head column
Binding[["Head", "Head_ID"]] = Binding[["Head_ID", "Head"]]

Binding["Head_detail_name"] = Binding["Head"].map(Pubchem_IUPAC_CID_Dict)
Binding["Head_SMILE_name"]  = Binding["Head"].map(Pubchem_CID_Smile_Dict)
Binding["Tail_detail_name"] = Binding["Tail"].map(Uniprot_Recname_dictRev)

Binding = Binding[Binding["Head_detail_name"].notna()]
Binding = Binding[Binding["Tail_detail_name"].notna()]

Binding["Head_ID_IS"] = "Pubchem"
Binding["Tail_ID_IS"] = "Uniprot_ID"
Binding["Head_type"]  = "ChemicalEntity"
Binding["Tail_type"]  = "Protein"
Binding["Relation"]   = Binding["Head_type"] + "_" + Binding["Tail_type"]
Binding["KG_Source"]  = "BindingDB"  # "BindingDB_MetaboGlue"

desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "KG_Source", "Species", "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
]
existing_cols = [c for c in desired_cols if c in Binding.columns]
Binding = Binding[existing_cols]
print(f"Final BindingDB rows: {len(Binding):,}")
Binding


/tmp/ipykernel_2250426/983809185.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Binding[["Head", "Head_ID"]] = Binding[["Head_ID", "Head"]]
/tmp/ipykernel_2250426/983809185.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Binding["Head_detail_name"] = Binding["Head"].map(Pubchem_IUPAC_CID_Dict)
/tmp/ipykernel_2250426/983809185.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in t

Final BindingDB rows: 220,939


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Species,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,3009304,ChemicalEntity_Protein,P35968,ChemicalEntity,Protein,BindingDB,Homo sapiens,"6-[(4R,5S,6S,7R)-4,7-dibenzyl-3-(5-carboxypent...",C1=CC=C(C=C1)C[C@@H]2[C@@H]([C@H]([C@H](N(C(=O...,Vascular endothelial growth factor receptor 2 ...,Pubchem,Uniprot_ID
96,44520983,ChemicalEntity_Protein,P0DMS8,ChemicalEntity,Protein,BindingDB,Homo sapiens,2-amino-N-[(1-ethylimidazol-2-yl)methyl]-6-(5-...,CCN1C=CN=C1CNC(=O)C2=NC(=NC(=C2)C3=CC=C(O3)C)N,Adenosine receptor A3,Pubchem,Uniprot_ID
139,49831257,ChemicalEntity_Protein,P52333,ChemicalEntity,Protein,BindingDB,Homo sapiens,"N-[5-[4-[(1,1-dioxo-1,4-thiazinan-4-yl)methyl]...",C1CC1C(=O)NC2=NN3C(=N2)C=CC=C3C4=CC=C(C=C4)CN5...,Tyrosine-protein kinase JAK3 {ECO:0000305},Pubchem,Uniprot_ID
141,72712978,ChemicalEntity_Protein,P36888,ChemicalEntity,Protein,BindingDB,Homo sapiens,8-(3-aminophenyl)-2-(4-methoxyanilino)pteridin...,COC1=CC=C(C=C1)NC2=NC=C3C(=N2)N(C(=O)C=N3)C4=C...,Receptor-type tyrosine-protein kinase FLT3,Pubchem,Uniprot_ID
194,5327301,ChemicalEntity_Protein,P42575,ChemicalEntity,Protein,BindingDB,Homo sapiens,5-[[[5-[[(2S)-1-carboxy-4-[(2-chlorophenyl)met...,CN(CC1=CC(=C(C=C1)O)C(=O)O)CC2=CC=C(S2)C(=O)N[...,Caspase-2 subunit p12,Pubchem,Uniprot_ID
...,...,...,...,...,...,...,...,...,...,...,...,...
422157,145977670,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo sapiens,"1-(4-methylsulfonylphenyl)-3-(1H-pyrrolo[2,3-b...",CS(=O)(=O)C1=CC=C(C=C1)N2C3=NC=NC(=C3C(=N2)C4=...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID
422158,145977679,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo sapiens,"6-[4-amino-3-(1H-pyrrolo[2,3-b]pyridin-5-yl)py...",C1CC2=C(CC1N3C4=NC=NC(=C4C(=N3)C5=CN=C6C(=C5)C...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID
422159,118025908,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo sapiens,"1-(2,3-dihydro-1H-inden-2-yl)-3-(1H-pyrrolo[2,...",C1C(CC2=CC=CC=C21)N3C4=NC=NC(=C4C(=N3)C5=CN=C6...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID
422161,1363752,ChemicalEntity_Protein,Q99814,ChemicalEntity,Protein,BindingDB,Homo sapiens,3-[[2-(4-tert-butylphenoxy)acetyl]amino]benzoi...,CC(C)(C)C1=CC=C(C=C1)OCC(=O)NC2=CC=CC(=C2)C(=O)O,Endothelial PAS domain-containing protein 1,Pubchem,Uniprot_ID


In [1]:
Binding

NameError: name 'Binding' is not defined

In [23]:
Binding["KG_Source"]  = "BindingDB"

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


NameError: name 'Binding' is not defined

In [22]:
Binding_Human = Binding[Binding["Species"] == "Homo sapiens"]
Binding_Mouse = Binding[Binding["Species"] == "Mus musculus"]
Binding_Yeast = Binding[Binding["Species"] == "Saccharomyces cerevisiae"]
Binding_Cele  = Binding[Binding["Species"] == "Caenorhabditis elegans"]
Binding_Droso = Binding[Binding["Species"] == "Drosophila melanogaster"]
Binding_Zebra = Binding[Binding["Species"] == "Danio rerio"]

Binding_Human.to_csv(os.path.join(OUT_PATH, "bindingdb/Chemical_Protein_Human_BindingDB.csv"),  index=False)
Binding_Mouse.to_csv(os.path.join(OUT_PATH, "bindingdb/Chemical_Protein_Mouse_BindingDB.csv"),  index=False)
Binding_Yeast.to_csv(os.path.join(OUT_PATH, "bindingdb/Chemical_Protein_Yeast_BindingDB.csv"),  index=False)
Binding_Cele.to_csv( os.path.join(OUT_PATH, "bindingdb/Chemical_Protein_Cele_BindingDB.csv"),   index=False)
Binding_Droso.to_csv(os.path.join(OUT_PATH, "bindingdb/Chemical_Protein_Droso_BindingDB.csv"),  index=False)
Binding_Zebra.to_csv(os.path.join(OUT_PATH, "bindingdb/Chemical_Protein_Zebra_BindingDB.csv"),  index=False)
print("BindingDB: all 6 species files saved.")
del Binding, Binding_Human, Binding_Mouse, Binding_Yeast, Binding_Cele, Binding_Droso, Binding_Zebra


BindingDB: all 6 species files saved.


In [25]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/'

In [28]:
pd.read_csv(f"{OUT_PATH}/bindingdb/Chemical_Protein_Human_BindingDB.csv")

,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Species,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,3009304,ChemicalEntity_Protein,P35968,ChemicalEntity,Protein,BindingDB,Homo sapiens,"6-[(4R,5S,6S,7R)-4,7-dibenzyl-3-(5-carboxypent...",C1=CC=C(C=C1)C[C@@H]2[C@@H]([C@H]([C@H](N(C(=O...,Vascular endothelial growth factor receptor 2 ...,Pubchem,Uniprot_ID
1,44520983,ChemicalEntity_Protein,P0DMS8,ChemicalEntity,Protein,BindingDB,Homo sapiens,2-amino-N-[(1-ethylimidazol-2-yl)methyl]-6-(5-...,CCN1C=CN=C1CNC(=O)C2=NC(=NC(=C2)C3=CC=C(O3)C)N,Adenosine receptor A3,Pubchem,Uniprot_ID
2,49831257,ChemicalEntity_Protein,P52333,ChemicalEntity,Protein,BindingDB,Homo sapiens,"N-[5-[4-[(1,1-dioxo-1,4-thiazinan-4-yl)methyl]...",C1CC1C(=O)NC2=NN3C(=N2)C=CC=C3C4=CC=C(C=C4)CN5...,Tyrosine-protein kinase JAK3 {ECO:0000305},Pubchem,Uniprot_ID
3,72712978,ChemicalEntity_Protein,P36888,ChemicalEntity,Protein,BindingDB,Homo sapiens,8-(3-aminophenyl)-2-(4-methoxyanilino)pteridin...,COC1=CC=C(C=C1)NC2=NC=C3C(=N2)N(C(=O)C=N3)C4=C...,Receptor-type tyrosine-protein kinase FLT3,Pubchem,Uniprot_ID
4,5327301,ChemicalEntity_Protein,P42575,ChemicalEntity,Protein,BindingDB,Homo sapiens,5-[[[5-[[(2S)-1-carboxy-4-[(2-chlorophenyl)met...,CN(CC1=CC(=C(C=C1)O)C(=O)O)CC2=CC=C(S2)C(=O)N[...,Caspase-2 subunit p12,Pubchem,Uniprot_ID
...,...,...,...,...,...,...,...,...,...,...,...,...
207848,145977670,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo sapiens,"1-(4-methylsulfonylphenyl)-3-(1H-pyrrolo[2,3-b...",CS(=O)(=O)C1=CC=C(C=C1)N2C3=NC=NC(=C3C(=N2)C4=...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID
207849,145977679,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo sapiens,"6-[4-amino-3-(1H-pyrrolo[2,3-b]pyridin-5-yl)py...",C1CC2=C(CC1N3C4=NC=NC(=C4C(=N3)C5=CN=C6C(=C5)C...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID
207850,118025908,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo sapiens,"1-(2,3-dihydro-1H-inden-2-yl)-3-(1H-pyrrolo[2,...",C1C(CC2=CC=CC=C21)N3C4=NC=NC(=C4C(=N3)C5=CN=C6...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID
207851,1363752,ChemicalEntity_Protein,Q99814,ChemicalEntity,Protein,BindingDB,Homo sapiens,3-[[2-(4-tert-butylphenoxy)acetyl]amino]benzoi...,CC(C)(C)C1=CC=C(C=C1)OCC(=O)NC2=CC=CC(=C2)C(=O)O,Endothelial PAS domain-containing protein 1,Pubchem,Uniprot_ID


## 5. SMS — Small Molecule Suite (Harvard)

In [48]:
SMS = pd.read_csv(os.path.join(BASE_PATH, "sms/SMS.csv"))
print(f"Raw SMS rows: {len(SMS):,}")

SMS.rename(columns={"chembl_id": "Head", "symbol": "Tail"}, inplace=True)

Raw SMS rows: 1,537,113


In [49]:
# Bug 1 fixed: reuse shared map_head_id()
tqdm.pandas(desc="SMS – mapping Head → PubChem CID")
SMS["Head_ID"] = SMS["Head"].progress_apply(map_head_id)
SMS["Head_ID"] = SMS["Head_ID"].astype(str).str.replace(r"\.0$", "", regex=True)
SMS = SMS[SMS["Head_ID"].notna() & (SMS["Head_ID"] != "nan")]
print(f"After CID mapping: {len(SMS):,}")


SMS – mapping Head → PubChem CID: 100%|██████████| 1537113/1537113 [00:02<00:00, 711689.28it/s]


After CID mapping: 1,508,889


In [50]:
SMS

,lspci_id,lspci_target_id,source,description_assay,gene_id,Tail,Head,pref_name,inchi,Head_ID
0,2,833,chembl,Binding affinity at heteropentameric Nicotinic...,1137.0,CHRNA4,CHEMBL46486,TETRAMETHYLAMMONIUM ION,"InChI=1S/C4H12N/c1-5(2,3)4/h1-4H3/q+1",6380
1,2,835,chembl,Binding affinity at homopentameric Nicotinic a...,1139.0,CHRNA7,CHEMBL46486,TETRAMETHYLAMMONIUM ION,"InChI=1S/C4H12N/c1-5(2,3)4/h1-4H3/q+1",6380
2,2,837,chembl,Binding affinity at heteropentameric Nicotinic...,1141.0,CHRNB2,CHEMBL46486,TETRAMETHYLAMMONIUM ION,"InChI=1S/C4H12N/c1-5(2,3)4/h1-4H3/q+1",6380
3,2,30845,chembl,Binding affinity at homopentameric Nicotinic a...,89832.0,CHRFAM7A,CHEMBL46486,TETRAMETHYLAMMONIUM ION,"InChI=1S/C4H12N/c1-5(2,3)4/h1-4H3/q+1",6380
4,4,43673,chembl,Compound was evaluated for reversible inhibiti...,290567.0,Chat,CHEMBL439723,TRIMETHYLAMMONIUM,InChI=1S/C3H9N/c1-4(2)3/h1-3H3,1146
...,...,...,...,...,...,...,...,...,...,...
1537108,26169000,3876,chembl,Inhibition of 20S immunoproteasome beta 5 in h...,5696.0,PSMB8,CHEMBL4590398,NaN,InChI=1S/C31H43N7O6S/c1-20(2)16-26(35-30(41)27...,155567712
1537109,26169000,3878,chembl,Inhibition of 20S immunoproteasome beta 1 in h...,5698.0,PSMB9,CHEMBL4590398,NaN,InChI=1S/C31H43N7O6S/c1-20(2)16-26(35-30(41)27...,155567712
1537110,26169000,3878,chembl,Inhibition of 20S immunoproteasome beta 1 in h...,5698.0,PSMB9,CHEMBL4590398,NaN,InChI=1S/C31H43N7O6S/c1-20(2)16-26(35-30(41)27...,155567712
1537111,26169000,3879,chembl,Inhibition of 20S immunoproteasome beta 2 in h...,5699.0,PSMB10,CHEMBL4590398,NaN,InChI=1S/C31H43N7O6S/c1-20(2)16-26(35-30(41)27...,155567712


In [51]:
SMS[["Head", "Head_ID"]] = SMS[["Head_ID", "Head"]]

SMS["Head_detail_name"] = SMS["Head"].map(Pubchem_IUPAC_CID_Dict)
SMS["Head_SMILE_name"]  = SMS["Head"].map(Pubchem_CID_Smile_Dict)
SMS["Tail_detail_name"] = SMS["Tail"].map(NCBI_Synbol_GENEname_dict)

SMS = SMS[SMS["Head_detail_name"].notna()]
SMS = SMS[SMS["Tail_detail_name"].notna()]

SMS["Head_ID_IS"] = "Pubchem"
SMS["Tail_ID_IS"] = "NCBI_ID"   # SMS Tail is gene symbol resolved via Uniprot
SMS["Head_type"]  = "ChemicalEntity"
SMS["Tail_type"]  = "Gene"
SMS["Relation"]   = SMS["Head_type"] + "_" + SMS["Tail_type"]
SMS["KG_Source"]  = "SMS"

desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "KG_Source", "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
]
existing_cols = [c for c in desired_cols if c in SMS.columns]
SMS = SMS[existing_cols]
print(f"Final SMS rows: {len(SMS):,}")
SMS

Final SMS rows: 1,277,079


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,6380,ChemicalEntity_Gene,CHRNA4,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,cholinergic receptor nicotinic alpha 4 subunit,Pubchem,NCBI_ID
1,6380,ChemicalEntity_Gene,CHRNA7,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,cholinergic receptor nicotinic alpha 7 subunit,Pubchem,NCBI_ID
2,6380,ChemicalEntity_Gene,CHRNB2,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,cholinergic receptor nicotinic beta 2 subunit,Pubchem,NCBI_ID
3,6380,ChemicalEntity_Gene,CHRFAM7A,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,CHRNA7 (exons 5-10) and FAM7A (exons A-E) fusion,Pubchem,NCBI_ID
7,892,ChemicalEntity_Gene,APP,ChemicalEntity,Gene,SMS,"cyclohexane-1,2,3,4,5,6-hexol",C1(C(C(C(C(C1O)O)O)O)O)O,amyloid beta precursor protein,Pubchem,NCBI_ID
...,...,...,...,...,...,...,...,...,...,...,...
1537108,155567712,ChemicalEntity_Gene,PSMB8,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 8,Pubchem,NCBI_ID
1537109,155567712,ChemicalEntity_Gene,PSMB9,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 9,Pubchem,NCBI_ID
1537110,155567712,ChemicalEntity_Gene,PSMB9,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 9,Pubchem,NCBI_ID
1537111,155567712,ChemicalEntity_Gene,PSMB10,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 10,Pubchem,NCBI_ID


In [53]:
SMS.to_csv(os.path.join(OUT_PATH, "sms/Chemical_Gene_Human_SMS_Metaboglue.csv"), index=False)
print("SMS: saved.")
del SMS

NameError: name 'SMS' is not defined

## 6. ChEMBL

In [55]:
species_list = [
    "Drosophila melanogaster", "Mus musculus", "Saccharomyces cerevisiae",
    "Caenorhabditis elegans", "Danio rerio", "Homo sapiens", "Humans",
]

Chembl = pd.read_csv(os.path.join(BASE_PATH, "chembl/compound_info.csv"))
print(f"Raw ChEMBL rows: {len(Chembl):,}")

Chembl = Chembl[Chembl["Target Organism"].isin(species_list)]
print(f"After species filter: {len(Chembl):,}")

Chembl.rename(columns={
    "Molecule ChEMBL ID":  "Head",
    "Target Preferred Name": "Tail",
    "Target Organism":       "Species",
    "Canonical SMILES":      "Head_SEQ",
}, inplace=True)

Chembl["Relation"]        = "Chemical_Protein"
Chembl["Head_type"]       = "Chemical"
Chembl["Tail_type"]       = "Protein"
Chembl["Relation_Source"] = "Chembl"

Chembl = Chembl[[
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "Relation_Source", "Species", "Head_SEQ",
]]
Chembl


/tmp/ipykernel_1678776/3403066902.py:6: DtypeWarning: Columns (0,5,8,11,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  Chembl = pd.read_csv(os.path.join(BASE_PATH, "chembl/compound_info.csv"))


Raw ChEMBL rows: 14,606,181
After species filter: 7,101,976


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_SEQ
0,CHEMBL2114210,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OC[C@H]1N[C@H](CO)[C@@H](O)C(O)[C@@H]1O
1,CHEMBL2114210,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OC[C@H]1N[C@H](CO)[C@@H](O)C(O)[C@@H]1O
2,CHEMBL87169,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OCC1NC(CO)[C@@H](O)C(O)C1O
3,CHEMBL87169,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OCC1NC(CO)[C@@H](O)C(O)C1O
4,CHEMBL2115215,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OC[C@@H]1N[C@H](CO)[C@@H](O)[C@H](O)[C@@H]1O
...,...,...,...,...,...,...,...,...
14606176,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...
14606177,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...
14606178,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...
14606179,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...


In [56]:

tqdm.pandas(desc="ChEMBL – mapping Head → PubChem CID")
Chembl["Head_ID"] = Chembl["Head"].progress_apply(map_head_id)
Chembl["Head_ID"] = Chembl["Head_ID"].astype(str).str.replace(r"\.0$", "", regex=True)
Chembl = Chembl[Chembl["Head_ID"].notna() & (Chembl["Head_ID"] != "nan")]
print(f"After CID mapping: {len(Chembl):,}")


ChEMBL – mapping Head → PubChem CID: 100%|██████████| 7101976/7101976 [00:08<00:00, 803118.17it/s] 


After CID mapping: 6,829,507


In [57]:
Chembl

,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_SEQ,Head_ID
0,CHEMBL2114210,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OC[C@H]1N[C@H](CO)[C@@H](O)C(O)[C@@H]1O,195553
1,CHEMBL2114210,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OC[C@H]1N[C@H](CO)[C@@H](O)C(O)[C@@H]1O,195553
2,CHEMBL87169,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OCC1NC(CO)[C@@H](O)C(O)C1O,44320078
3,CHEMBL87169,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OCC1NC(CO)[C@@H](O)C(O)C1O,44320078
4,CHEMBL2115215,Chemical_Protein,Maltase-glucoamylase,Chemical,Protein,Chembl,Homo sapiens,OC[C@@H]1N[C@H](CO)[C@@H](O)[C@H](O)[C@@H]1O,6475029
...,...,...,...,...,...,...,...,...,...
14606176,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...,11213350
14606177,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...,11213350
14606178,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...,11213350
14606179,CHEMBL375563,Chemical_Protein,SF-188,Chemical,Protein,Chembl,Homo sapiens,CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H...,11213350


In [58]:
Chembl[["Head", "Head_ID"]] = Chembl[["Head_ID", "Head"]]

Chembl["Head_detail_name"] = Chembl["Head"].map(Pubchem_IUPAC_CID_Dict)
Chembl["Head_SMILE_name"]  = Chembl["Head"].map(Pubchem_CID_Smile_Dict)

# Map Target Preferred Name → NCBI Gene Symbol
Chembl["Tail_ID"] = Chembl["Tail"].map(NCBI_GENEname__Symbol_dict)
Chembl = Chembl[Chembl["Tail_ID"].notna()]

# Bug 2 fixed: was rename({"ABC": "Tail_detail_name"}) which silently did nothing.
# Correct approach: swap Tail ↔ Tail_ID so canonical symbol goes into Tail,
# then annotate Tail_detail_name from the symbol.
Chembl = Chembl.loc[:, ~Chembl.columns.duplicated(keep="last")]
Chembl[["Tail", "Tail_detail_name"]] = Chembl[["Tail_ID", "Tail"]]
# Now Tail = gene symbol, Tail_detail_name = original preferred name string

Chembl = Chembl[Chembl["Head_detail_name"].notna()]
Chembl = Chembl[Chembl["Tail_detail_name"].notna()]

Chembl["Head_ID_IS"] = "Pubchem"
Chembl["Tail_ID_IS"] = "NCBI_ID"
Chembl["Head_type"]  = "ChemicalEntity"
Chembl["Tail_type"]  = "Gene"
Chembl["Relation"]   = Chembl["Head_type"] + "_" + Chembl["Tail_type"]
Chembl["KG_Source"]  = "Chembl_MetaboGlue"

desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "KG_Source", "Species", "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
]
existing_cols = [c for c in desired_cols if c in Chembl.columns]
Chembl = Chembl[existing_cols]
print(f"Final ChEMBL rows: {len(Chembl):,}")
Chembl

/tmp/ipykernel_1678776/1677159765.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Chembl[["Head", "Head_ID"]] = Chembl[["Head_ID", "Head"]]
/tmp/ipykernel_1678776/1677159765.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Chembl["Head_detail_name"] = Chembl["Head"].map(Pubchem_IUPAC_CID_Dict)
/tmp/ipykernel_1678776/1677159765.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Final ChEMBL rows: 77,802


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Species,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
216530,44342104,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,"6-(4-bromoanilino)-7-chloroquinazoline-5,8-dione",C1=CC(=CC=C1NC2=C(C(=O)C3=NC=NC=C3C2=O)Cl)Br,DNA topoisomerase I,Pubchem,NCBI_ID
216531,10405313,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,"7-chloro-6-(2,4-dimethoxyanilino)quinazoline-5...",COC1=CC(=C(C=C1)NC2=C(C(=O)C3=NC=NC=C3C2=O)Cl)OC,DNA topoisomerase I,Pubchem,NCBI_ID
216532,5472243,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,"(11E)-11-(4-chlorobutylidene)-15,16-dimethoxy-...",CN1C2=C(C3=CC(=C(C=C3C1=O)OC)OC)/C(=C/CCCCl)/C...,DNA topoisomerase I,Pubchem,NCBI_ID
216533,10244294,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,2-(3H-benzimidazol-5-yl)-3H-benzimidazole-5-ca...,C1=CC2=C(C=C1C#N)NC(=N2)C3=CC4=C(C=C3)N=CN4,DNA topoisomerase I,Pubchem,NCBI_ID
216534,44340518,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,"(11Z)-11-(4-iodobutylidene)-15,16-dimethoxy-20...",CN1C2=C(C3=CC(=C(C=C3C1=O)OC)OC)/C(=C\CCCI)/C4...,DNA topoisomerase I,Pubchem,NCBI_ID
...,...,...,...,...,...,...,...,...,...,...,...,...
8903275,4371240,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,N-benzyl-4-(benzylsulfamoyl)benzamide,C1=CC=C(C=C1)CNC(=O)C2=CC=C(C=C2)S(=O)(=O)NCC3...,LIM domain kinase 1,Pubchem,NCBI_ID
8903276,137796294,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,N-benzyl-N-butyl-4-(phenylsulfamoyl)benzamide,CCCCN(CC1=CC=CC=C1)C(=O)C2=CC=C(C=C2)S(=O)(=O)...,LIM domain kinase 1,Pubchem,NCBI_ID
8903277,137796294,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,N-benzyl-N-butyl-4-(phenylsulfamoyl)benzamide,CCCCN(CC1=CC=CC=C1)C(=O)C2=CC=C(C=C2)S(=O)(=O)...,LIM domain kinase 1,Pubchem,NCBI_ID
8903278,137796294,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl_MetaboGlue,Homo sapiens,N-benzyl-N-butyl-4-(phenylsulfamoyl)benzamide,CCCCN(CC1=CC=CC=C1)C(=O)C2=CC=C(C=C2)S(=O)(=O)...,LIM domain kinase 1,Pubchem,NCBI_ID


In [59]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/'

In [60]:
Chembl.to_csv(os.path.join(OUT_PATH, "chembl/Chemical_Gene_Human_Chembl_Metaboglue.csv"), index=False)
print("ChEMBL: saved.")
# del Chembl


ChEMBL: saved.


## 7. BioSNAP — ChG Miner (DrugBank Targets)

In [61]:
BASE_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

In [63]:
# Load DrugBank for Target_Id → Target_Name lookup
_drugbank_raw = pd.read_csv(os.path.join(BASE_PATH, "drugbank/Drugbank.csv"))
DrugBank_dict = dict(zip(_drugbank_raw["Target_Id"], _drugbank_raw["Target_Name"]))
del _drugbank_raw

BioSNAP = pd.read_csv(os.path.join(BASE_PATH, "biosnap/Biosnap_ChG_Miner.csv"))
BioSNAP["Target"] = BioSNAP["Target"].astype(str).str.split(":").str[0]
BioSNAP["Target"] = BioSNAP["Target"].map(DrugBank_dict)
print(f"Raw BioSNAP rows: {len(BioSNAP):,}")

BioSNAP.rename(columns={
    "Database_Id": "Head",
    "compound":    "Head_name",
    "Target":      "Tail",
    "SMILES":      "Head_SEQ",
}, inplace=True)

BioSNAP["Relation"]        = "Chemical_Protein"
BioSNAP["Head_type"]       = "Chemical"
BioSNAP["Tail_type"]       = "Protein"
BioSNAP["Species"]         = "Homo sapiens"
BioSNAP["Relation_Source"] = "BioSNAP_MetaboGlue"

BioSNAP = BioSNAP[[
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "Relation_Source", "Species", "Head_name", "Head_SEQ",
]]
BioSNAP


Raw BioSNAP rows: 13,829


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_name,Head_SEQ
0,DB00357,Chemical_Protein,"Cholesterol side-chain cleavage enzyme, mitoch...",Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,Aminoglutethimide,CCC1(CCC(=O)NC1=O)C1=CC=C(N)C=C1
1,DB00357,Chemical_Protein,"Cholesterol side-chain cleavage enzyme, mitoch...",Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,Aminoglutethimide,CCC1(CCC(=O)NC1=O)C1=CC=C(N)C=C1
2,DB00357,Chemical_Protein,"Cholesterol side-chain cleavage enzyme, mitoch...",Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,Aminoglutethimide,CCC1(CCC(=O)NC1=O)C1=CC=C(N)C=C1
3,DB00357,Chemical_Protein,"Cholesterol side-chain cleavage enzyme, mitoch...",Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,Aminoglutethimide,CCC1(CCC(=O)NC1=O)C1=CC=C(N)C=C1
4,DB00357,Chemical_Protein,"Cholesterol side-chain cleavage enzyme, mitoch...",Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,Aminoglutethimide,CCC1(CCC(=O)NC1=O)C1=CC=C(N)C=C1
...,...,...,...,...,...,...,...,...,...
13824,DB00235,Chemical_Protein,"cGMP-inhibited 3',5'-cyclic phosphodiesterase A",Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,Milrinone,CC1=C(C=C(C#N)C(=O)N1)C1=CC=NC=C1
13825,DB04249,Chemical_Protein,Cytochrome c,Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,Zinc Substituted Heme C,[Zn+4].[H]\C(C)=C1\C(C)=C2[N-]\C\1=C([H])/C1=C...
13826,DB06883,Chemical_Protein,Proto-oncogene tyrosine-protein kinase Src,Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,1-[1-(3-aminophenyl)-3-tert-butyl-1H-pyrazol-5...,CC(C)(C)C1=NN(C(NC(=O)NC2=CC=CC=C2)=C1)C1=CC=C...
13827,DB08365,Chemical_Protein,Wee1-like protein kinase,Chemical,Protein,BioSNAP_MetaboGlue,Homo sapiens,8-bromo-4-(2-chlorophenyl)-N-(2-hydroxyethyl)-...,CN1C2=C(C(Br)=C1C(=O)NCCO)C1=C(C(=O)NC1=O)C(=C...


In [64]:
BioSNAP["Head_ID"] = BioSNAP["Head"].map(DB2PC_dict).fillna(BioSNAP["Head"])
BioSNAP["Head_ID"] = BioSNAP["Head_ID"].astype(str).str.replace(r"\.0$", "", regex=True)
BioSNAP = BioSNAP[BioSNAP["Head_ID"].notna() & (BioSNAP["Head_ID"] != "nan")]

BioSNAP["Tail_ID"] = BioSNAP["Tail"].map(Uniprot_Recname_dict)
BioSNAP = BioSNAP[BioSNAP["Tail_ID"].notna()]

BioSNAP[["Head", "Head_ID"]] = BioSNAP[["Head_ID", "Head"]]
BioSNAP[["Tail", "Tail_ID"]] = BioSNAP[["Tail_ID", "Tail"]]

BioSNAP["Head_detail_name"] = BioSNAP["Head"].map(Pubchem_IUPAC_CID_Dict)
BioSNAP["Head_SMILE_name"]  = BioSNAP["Head"].map(Pubchem_CID_Smile_Dict)
BioSNAP["Tail_detail_name"] = BioSNAP["Tail"].map(Uniprot_Recname_dictRev)

BioSNAP = BioSNAP[BioSNAP["Head_detail_name"].notna()]
BioSNAP = BioSNAP[BioSNAP["Tail_detail_name"].notna()]

BioSNAP["Head_ID_IS"] = "Pubchem"
BioSNAP["Tail_ID_IS"] = "Uniprot_ID"
BioSNAP["Head_type"]  = "ChemicalEntity"
BioSNAP["Tail_type"]  = "Protein"
BioSNAP["Relation"]   = BioSNAP["Head_type"] + "_" + BioSNAP["Tail_type"]
BioSNAP["KG_Source"]  = "BioSNAP"   # Bug 6 (BioSNAP1): was BindingDB_MetaboGlue

desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "KG_Source", "Species", "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
]
existing_cols = [c for c in desired_cols if c in BioSNAP.columns]
BioSNAP = BioSNAP[existing_cols]
print(f"Final BioSNAP rows: {len(BioSNAP):,}")
BioSNAP


Final BioSNAP rows: 9,766


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Species,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID
1,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID
2,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID
3,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID
4,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID
...,...,...,...,...,...,...,...,...,...,...,...,...
13823,17429,ChemicalEntity_Protein,O97959,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,piperidine-1-carbaldehyde,C1CCN(CC1)C=O,Alcohol dehydrogenase 1C,Pubchem,Uniprot_ID
13825,131704274,ChemicalEntity_Protein,Q6C9Q0,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,"3-[(1Z,4Z,8E,9Z,13Z,14Z)-18-(2-carboxyethyl)-8...",C/C=C\1/C(=C2/C=C\3/C(=C/C)/C(=C([N-]3)/C=C\4/...,Cytochrome c,Pubchem,Uniprot_ID
13826,10915062,ChemicalEntity_Protein,Q9JJ10,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,1-[2-(3-aminophenyl)-5-tert-butylpyrazol-3-yl]...,CC(C)(C)C1=NN(C(=C1)NC(=O)NC2=CC=CC=C2)C3=CC=C...,Proto-oncogene tyrosine-protein kinase Src,Pubchem,Uniprot_ID
13827,11691442,ChemicalEntity_Protein,Q5EAN3,ChemicalEntity,Protein,BioSNAP_MetaboGlue,Homo sapiens,8-bromo-4-(2-chlorophenyl)-N-(2-hydroxyethyl)-...,CN1C2=C(C3=C(C(=C2)C4=CC=CC=C4Cl)C(=O)NC3=O)C(...,Wee1-like protein kinase,Pubchem,Uniprot_ID


In [66]:
OUT_PATH
BioSNAP["KG_Source"]  = "BioSNAP" 

In [67]:
BioSNAP.to_csv(os.path.join(OUT_PATH, "biosnap/Chemical_Protein_BioSNAP_Metaboglue.csv"), index=False)
print("BioSNAP: saved.")
del BioSNAP

BioSNAP: saved.


## 8. BioSNAP 2 — Target Decagon (Drug–Gene)



In [70]:
biosnap2 = pd.read_csv(os.path.join(BASE_PATH, "biosnap/Target_Decagon.csv"))
biosnap2 = biosnap2[~biosnap2["SMILE "].isna()]
print(f"Raw BioSNAP-2 rows: {len(biosnap2):,}")

biosnap2.rename(columns={
    "PubChEM CID ": "Head",
    "Name ":         "Head_name",
    "name":          "Tail_name",
    "symbol":        "Tail",
    "SMILE ":        "Head_SEQ",
}, inplace=True)

# biosnap2["Relation"]        = "Gene_Protein"
# biosnap2["Head_type"]       = "Gene"
# biosnap2["Tail_type"]       = "Protein"
# biosnap2["Species"]         = "Homo sapiens"
# biosnap2["Relation_Source"] = "BioSNAP_MetaboGlue"

# biosnap2 = biosnap2[[
#     "Head", "Relation", "Tail", "Head_type", "Tail_type",
#     "Relation_Source", "Species", "Head_name", "Head_SEQ", "Tail_name",
# ]]
biosnap2


Raw BioSNAP-2 rows: 65,339


,Head,Head_name,Head_SEQ,Chemical,Gene,query,entrezgene,Tail_name,Tail,alias,taxid,HGNC
25,CID100003355,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,CID100003355,3757,3757,3757.0,potassium voltage-gated channel subfamily H me...,KCNH2,"ERG-1, ERG1, H-ERG, HERG, HERG1",9606.0,HGNC:6251
26,CID100003355,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,CID100003355,6331,6331,6331.0,sodium voltage-gated channel alpha subunit 5,SCN5A,"CDCD2, CMD1E, CMPD2, HB1, HB2",9606.0,HGNC:10593
27,CID100003510,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,CID100003510,3757,3757,3757.0,potassium voltage-gated channel subfamily H me...,KCNH2,"ERG-1, ERG1, H-ERG, HERG, HERG1",9606.0,HGNC:6251
28,CID100003510,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,CID100003510,3359,3359,3359.0,5-hydroxytryptamine receptor 3A,HTR3A,"5-HT-3, 5-HT3A, 5-HT3R, 5HT3R, HTR3",9606.0,HGNC:5297
29,CID100003510,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,CID100003510,3363,3363,3363.0,5-hydroxytryptamine receptor 7,HTR7,5-HT7,9606.0,HGNC:5302
...,...,...,...,...,...,...,...,...,...,...,...,...
67852,CID100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,93589,93589,93589.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D4,RCD4,9606.0,HGNC:20202
67853,CID100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,9254,9254,9254.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D2,"CACNA2D, CASVDD",9606.0,HGNC:1400
67854,CID100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,55799,55799,55799.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D3,HSA272268,9606.0,HGNC:15460
67855,CID100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,781,781,781.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D1,"CACNA2, CACNL2A, CCHL2A, DEE110, LINC01112",9606.0,HGNC:1399


In [71]:
def clean_chemical_id(chem_id):
    """Strip leading 'CID' prefix and leading zeros from PubChem CIDs."""
    return str(chem_id).lstrip("CID").lstrip("0")

biosnap2["Head"] = biosnap2["Head"].map(clean_chemical_id)
biosnap2


,Head,Head_name,Head_SEQ,Chemical,Gene,query,entrezgene,Tail_name,Tail,alias,taxid,HGNC
25,100003355,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,CID100003355,3757,3757,3757.0,potassium voltage-gated channel subfamily H me...,KCNH2,"ERG-1, ERG1, H-ERG, HERG, HERG1",9606.0,HGNC:6251
26,100003355,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,CID100003355,6331,6331,6331.0,sodium voltage-gated channel alpha subunit 5,SCN5A,"CDCD2, CMD1E, CMPD2, HB1, HB2",9606.0,HGNC:10593
27,100003510,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,CID100003510,3757,3757,3757.0,potassium voltage-gated channel subfamily H me...,KCNH2,"ERG-1, ERG1, H-ERG, HERG, HERG1",9606.0,HGNC:6251
28,100003510,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,CID100003510,3359,3359,3359.0,5-hydroxytryptamine receptor 3A,HTR3A,"5-HT-3, 5-HT3A, 5-HT3R, 5HT3R, HTR3",9606.0,HGNC:5297
29,100003510,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,CID100003510,3363,3363,3363.0,5-hydroxytryptamine receptor 7,HTR7,5-HT7,9606.0,HGNC:5302
...,...,...,...,...,...,...,...,...,...,...,...,...
67852,100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,93589,93589,93589.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D4,RCD4,9606.0,HGNC:20202
67853,100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,9254,9254,9254.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D2,"CACNA2D, CASVDD",9606.0,HGNC:1400
67854,100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,55799,55799,55799.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D3,HSA272268,9606.0,HGNC:15460
67855,100125889,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,CID100125889,781,781,781.0,calcium voltage-gated channel auxiliary subuni...,CACNA2D1,"CACNA2, CACNL2A, CCHL2A, DEE110, LINC01112",9606.0,HGNC:1399


In [72]:
biosnap2["Head_detail_name"] = biosnap2["Head"].map(Pubchem_IUPAC_CID_Dict)
biosnap2["Head_SMILE_name"]  = biosnap2["Head"].map(Pubchem_CID_Smile_Dict)
biosnap2["Tail_detail_name"] = biosnap2["Tail"].map(NCBI_Synbol_GENEname_dict)

biosnap2 = biosnap2[biosnap2["Head_detail_name"].notna()]
biosnap2 = biosnap2[biosnap2["Tail_detail_name"].notna()]

biosnap2["Head_ID_IS"] = "Pubchem"
biosnap2["Tail_ID_IS"] = "NCBI_ID"         
biosnap2["Head_type"]  = "ChemicalEntity"
biosnap2["Tail_type"]  = "Gene"             
biosnap2["Relation"]   = biosnap2["Head_type"] + "_" + biosnap2["Tail_type"]
biosnap2["KG_Source"]  = "BioSNAP" 

desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "KG_Source", "Species", "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
]
existing_cols = [c for c in desired_cols if c in biosnap2.columns]
biosnap2 = biosnap2[existing_cols]
print(f"Final BioSNAP-2 rows: {len(biosnap2):,}")
biosnap2

Final BioSNAP-2 rows: 65,296


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
25,100003355,ChemicalEntity_Gene,KCNH2,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,potassium voltage-gated channel subfamily H me...,Pubchem,NCBI_ID
26,100003355,ChemicalEntity_Gene,SCN5A,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,sodium voltage-gated channel alpha subunit 5,Pubchem,NCBI_ID
27,100003510,ChemicalEntity_Gene,KCNH2,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,potassium voltage-gated channel subfamily H me...,Pubchem,NCBI_ID
28,100003510,ChemicalEntity_Gene,HTR3A,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,5-hydroxytryptamine receptor 3A,Pubchem,NCBI_ID
29,100003510,ChemicalEntity_Gene,HTR7,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,5-hydroxytryptamine receptor 7,Pubchem,NCBI_ID
...,...,...,...,...,...,...,...,...,...,...,...
67852,100125889,ChemicalEntity_Gene,CACNA2D4,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID
67853,100125889,ChemicalEntity_Gene,CACNA2D2,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID
67854,100125889,ChemicalEntity_Gene,CACNA2D3,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID
67855,100125889,ChemicalEntity_Gene,CACNA2D1,ChemicalEntity,Gene,BioSNAP_MetaboGlue,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID


In [73]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/'

In [74]:
biosnap2.to_csv(os.path.join(OUT_PATH, "biosnap/Chemical_Gene_BioSNAP_2_Metaboglue.csv"), index=False)
print("BioSNAP-2: saved.")
# del biosnap2


BioSNAP-2: saved.


## 9. DrugBank


In [79]:
species_list_db = [
    "Drosophila melanogaster", "Mus musculus", "Saccharomyces cerevisiae",
    "Caenorhabditis elegans", "Danio rerio", "Homo sapiens",
    "Zebrafish", "Mouse", "Humans", "Yeast", "Celegans", "Worm",
]

DrugBank = pd.read_csv(os.path.join(BASE_PATH, "drugbank/Drugbank.csv"))
print(f"Raw DrugBank rows: {len(DrugBank):,}")

DrugBank = DrugBank[DrugBank["organism"].isin(species_list_db)]
print(f"After species filter: {len(DrugBank):,}")
print(DrugBank["organism"].value_counts())


Raw DrugBank rows: 17,788
After species filter: 13,493
organism
Humans                    13446
Yeast                        39
Caenorhabditis elegans        4
Mouse                         4
Name: count, dtype: int64


In [80]:
DrugBank.rename(columns={
    "DrugBank Id":    "Head",
    "Target_Id":      "Tail_name",
    "Compound_Name":  "Head_name",
    "Target_Name":    "Tail",
    "Smiles":         "Head_SEQ",
    "organism":       "Species",
}, inplace=True)

DrugBank["Relation"]        = "Chemical_Protein"
DrugBank["Head_type"]       = "Chemical"
DrugBank["Tail_type"]       = "Protein"
DrugBank["Relation_Source"] = "DrugBank_MetaboGlue"

DrugBank = DrugBank[[
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "Relation_Source", "Species", "Head_name", "Head_SEQ", "Tail_name",
]]
DrugBank


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_name,Head_SEQ,Tail_name
0,DB00007,Chemical_Protein,Gonadotropin-releasing hormone receptor,Chemical,Protein,DrugBank_MetaboGlue,Humans,Leuprolide,CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(N)=N)NC(=...,BE0000203
1,DB00014,Chemical_Protein,Lutropin-choriogonadotropic hormone receptor,Chemical,Protein,DrugBank_MetaboGlue,Humans,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,BE0000134
2,DB00014,Chemical_Protein,Gonadotropin-releasing hormone receptor,Chemical,Protein,DrugBank_MetaboGlue,Humans,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,BE0000203
3,DB00035,Chemical_Protein,Vasopressin V2 receptor,Chemical,Protein,DrugBank_MetaboGlue,Humans,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,BE0000293
4,DB00035,Chemical_Protein,Vasopressin V1a receptor,Chemical,Protein,DrugBank_MetaboGlue,Humans,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,BE0000165
...,...,...,...,...,...,...,...,...,...,...
17781,DB16754,Chemical_Protein,Gamma-aminobutyric acid receptor subunit rho-3,Chemical,Protein,DrugBank_MetaboGlue,Humans,TPMPA,CP(O)(=O)C1=CCNCC1,BE0003619
17782,DB16756,Chemical_Protein,Tyrosine-protein kinase JAK1,Chemical,Protein,DrugBank_MetaboGlue,Humans,Ivarmacitinib,[H][C@@]12C[C@H](C[C@]1([H])CN(C2)C(=O)NC1=NC(...,BE0004145
17783,DB16778,Chemical_Protein,Prostate-specific antigen,Chemical,Protein,DrugBank_MetaboGlue,Humans,Lutetium Lu-177 vipivotide tetraxetan,[177Lu+3].OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=...,BE0008983
17784,DB16862,Chemical_Protein,Aryl hydrocarbon receptor,Chemical,Protein,DrugBank_MetaboGlue,Humans,Indigo,O=C1\C(NC2=C1C=CC=C2)=C1/NC2=C(C=CC=C2)C1=O,BE0003721


In [81]:
DrugBank["Head_ID"] = DrugBank["Head"].map(DB2PC_dict).fillna(DrugBank["Head"])
DrugBank["Head_ID"] = DrugBank["Head_ID"].astype(str).str.replace(r"\.0$", "", regex=True)
DrugBank = DrugBank[DrugBank["Head_ID"].notna() & (DrugBank["Head_ID"] != "nan")]

DrugBank["Tail_ID"] = DrugBank["Tail"].map(Uniprot_Recname_dict)
DrugBank = DrugBank[DrugBank["Tail_ID"].notna()]

DrugBank[["Head", "Head_ID"]] = DrugBank[["Head_ID", "Head"]]
DrugBank[["Tail", "Tail_ID"]] = DrugBank[["Tail_ID", "Tail"]]

DrugBank["Head_detail_name"] = DrugBank["Head"].map(Pubchem_IUPAC_CID_Dict)
DrugBank["Head_SMILE_name"]  = DrugBank["Head"].map(Pubchem_CID_Smile_Dict)
DrugBank["Tail_detail_name"] = DrugBank["Tail"].map(Uniprot_Recname_dictRev)
DrugBank

,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_name,Head_SEQ,Tail_name,Head_ID,Tail_ID,Head_detail_name,Head_SMILE_name,Tail_detail_name
0,657181,Chemical_Protein,Q9TTI8,Chemical,Protein,DrugBank_MetaboGlue,Humans,Leuprolide,CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(N)=N)NC(=...,BE0000203,DB00007,Gonadotropin-releasing hormone receptor,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCN=C(N)N)NC(=...,Gonadotropin-releasing hormone receptor
1,5311128,Chemical_Protein,Q28585,Chemical,Protein,DrugBank_MetaboGlue,Humans,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,BE0000134,DB00014,Lutropin-choriogonadotropic hormone receptor,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CC(C)C[C@@H](C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CC...,Lutropin-choriogonadotropic hormone receptor
2,5311128,Chemical_Protein,Q9TTI8,Chemical,Protein,DrugBank_MetaboGlue,Humans,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,BE0000203,DB00014,Gonadotropin-releasing hormone receptor,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CC(C)C[C@@H](C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CC...,Gonadotropin-releasing hormone receptor
3,5311065,Chemical_Protein,Q53ZC4,Chemical,Protein,DrugBank_MetaboGlue,Humans,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,BE0000293,DB00035,Vasopressin V2 receptor,(2S)-N-[(2R)-1-[(2-amino-2-oxoethyl)amino]-5-(...,C1C[C@H](N(C1)C(=O)[C@@H]2CSSCCC(=O)N[C@H](C(=...,Vasopressin V2 receptor
4,5311065,Chemical_Protein,P48043,Chemical,Protein,DrugBank_MetaboGlue,Humans,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,BE0000165,DB00035,Vasopressin V1a receptor,(2S)-N-[(2R)-1-[(2-amino-2-oxoethyl)amino]-5-(...,C1C[C@H](N(C1)C(=O)[C@@H]2CSSCCC(=O)N[C@H](C(=...,Vasopressin V1a receptor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17781,DB16754,Chemical_Protein,Q9UIV9,Chemical,Protein,DrugBank_MetaboGlue,Humans,TPMPA,CP(O)(=O)C1=CCNCC1,BE0003619,DB16754,Gamma-aminobutyric acid receptor subunit rho-3,NaN,NaN,Gamma-aminobutyric acid receptor subunit rho-3
17782,DB16756,Chemical_Protein,Q62126,Chemical,Protein,DrugBank_MetaboGlue,Humans,Ivarmacitinib,[H][C@@]12C[C@H](C[C@]1([H])CN(C2)C(=O)NC1=NC(...,BE0004145,DB16756,Tyrosine-protein kinase JAK1,NaN,NaN,Tyrosine-protein kinase JAK1
17783,DB16778,Chemical_Protein,P33619,Chemical,Protein,DrugBank_MetaboGlue,Humans,Lutetium Lu-177 vipivotide tetraxetan,[177Lu+3].OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=...,BE0008983,DB16778,Prostate-specific antigen,NaN,NaN,Prostate-specific antigen
17784,DB16862,Chemical_Protein,O89105,Chemical,Protein,DrugBank_MetaboGlue,Humans,Indigo,O=C1\C(NC2=C1C=CC=C2)=C1/NC2=C(C=CC=C2)C1=O,BE0003721,DB16862,Aryl hydrocarbon receptor,NaN,NaN,Aryl hydrocarbon receptor


In [83]:
DrugBank = DrugBank[DrugBank["Head_detail_name"].notna()]
DrugBank = DrugBank[DrugBank["Tail_detail_name"].notna()]

DrugBank["Head_ID_IS"] = np.where(
    DrugBank["Head"].isna(), None,
    np.where(DrugBank["Head"].astype(str).str.startswith("DB"), "Drugbank", "Pubchem")
)
# Bug 4 fixed: was BioSNAP['Tail_ID_IS'] — wrong DataFrame
DrugBank["Tail_ID_IS"] = "Uniprot_ID"
DrugBank["Head_type"]  = "ChemicalEntity"
DrugBank["Tail_type"]  = "Protein"
DrugBank["Relation"]   = DrugBank["Head_type"] + "_" + DrugBank["Tail_type"]
DrugBank["KG_Source"]  = "DrugBank_MetaboGlue"   # Bug 7 fixed: was BindingDB_MetaboGlue

desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "KG_Source", "Species", "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
]

existing_cols = [c for c in desired_cols if c in DrugBank.columns]
DrugBank = DrugBank[existing_cols]
print(f"Final DrugBank rows: {len(DrugBank):,}")
DrugBank

Final DrugBank rows: 10,139


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Species,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,657181,ChemicalEntity_Protein,Q9TTI8,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCN=C(N)N)NC(=...,Gonadotropin-releasing hormone receptor,Pubchem,Uniprot_ID
1,5311128,ChemicalEntity_Protein,Q28585,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CC(C)C[C@@H](C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CC...,Lutropin-choriogonadotropic hormone receptor,Pubchem,Uniprot_ID
2,5311128,ChemicalEntity_Protein,Q9TTI8,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CC(C)C[C@@H](C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CC...,Gonadotropin-releasing hormone receptor,Pubchem,Uniprot_ID
3,5311065,ChemicalEntity_Protein,Q53ZC4,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,(2S)-N-[(2R)-1-[(2-amino-2-oxoethyl)amino]-5-(...,C1C[C@H](N(C1)C(=O)[C@@H]2CSSCCC(=O)N[C@H](C(=...,Vasopressin V2 receptor,Pubchem,Uniprot_ID
4,5311065,ChemicalEntity_Protein,P48043,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,(2S)-N-[(2R)-1-[(2-amino-2-oxoethyl)amino]-5-(...,C1C[C@H](N(C1)C(=O)[C@@H]2CSSCCC(=O)N[C@H](C(=...,Vasopressin V1a receptor,Pubchem,Uniprot_ID
...,...,...,...,...,...,...,...,...,...,...,...,...
17670,137278711,ChemicalEntity_Protein,Q05147,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,6-fluoro-7-(2-fluoro-6-hydroxyphenyl)-1-(4-met...,C[C@H]1CN(CCN1C2=NC(=O)N(C3=NC(=C(C=C32)F)C4=C...,GTPase KRas,Pubchem,Uniprot_ID
17671,64945,ChemicalEntity_Protein,P23175,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,"(1S,2R,4aS,6aR,6aS,6bR,8aR,10S,12aR,14bS)-10-h...",C[C@@H]1CC[C@@]2(CC[C@@]3(C(=CC[C@H]4[C@]3(CC[...,GTPase HRas,Pubchem,Uniprot_ID
17672,64945,ChemicalEntity_Protein,Q61515,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,"(1S,2R,4aS,6aR,6aS,6bR,8aR,10S,12aR,14bS)-10-h...",C[C@@H]1CC[C@@]2(CC[C@@]3(C(=CC[C@H]4[C@]3(CC[...,Neutrophil elastase,Pubchem,Uniprot_ID
17675,169535,ChemicalEntity_Protein,O35172,ChemicalEntity,Protein,DrugBank_MetaboGlue,Humans,iron(3+);2-methyl-4-oxopyran-3-olate,CC1=C(C(=O)C=CO1)[O-].CC1=C(C(=O)C=CO1)[O-].CC...,Natural resistance-associated macrophage prote...,Pubchem,Uniprot_ID


In [84]:
DrugBank.to_csv(os.path.join(OUT_PATH, "drugbank/Chemical_Protein_DrugBank_Metaboglue.csv"), index=False)
print("DrugBank: saved.")
# del DrugBank

DrugBank: saved.


In [1]:
#